## Set up the environment

* verify attention thingy works and the same way as in keras
* change to transformer - ?
* veify the whole model code and forward func works
* masking func
* re dataloader and wrapper - copy it from the other model
* write func in dataloader that take batch and procduce inputs and outputs - shuffle etc.
* rerun data code with all movies

In [ ]:
# !pip install h5py==2.10.0 numpy==1.19.5 --force-reinstall

# Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# # navigate to ParlAI folder to import the functions
# %cd 'drive/My Drive/ParlAI/ParlAI'
%cd 'drive/MyDrive/The Movie Dataset'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/The Movie Dataset


In [ ]:
# # import dependencies
# from parlai.core.agents import Agent
# from parlai.core.params import ParlaiParser
# from parlai.core.worlds import DialogPartnerWorld
# import numpy as np
# import pandas as pd
# # %tensorflow_version 1.x
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM, Concatenate, Embedding, TimeDistributed, Reshape, Masking, Attention
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import ModelCheckpoint
# import random
from sklearn.metrics import log_loss
# import seaborn as sns
# import matplotlib.pyplot as plt
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import math
# # suppress TF future version compatibility warnings
# import os
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# import tensorflow as tf
# # tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)


# Build dataset

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import ast
from typing import Optional
from joblib import Memory
import os
import copy

In [ ]:
RATINGS_PATH = "ratings.csv"
RATINGS_SUBSET_PATH = 'ratings_subset.csv'
FINAL_DATASET_PATH = 'final_dataset.csv'
METADATA_PATH = "movies_metadata.csv"
CREDITS_PATH = "credits.csv"
KEYWORDS_PATH = "keywords.csv"
LINKS_PATH = "links.csv"
ATTRIBUTE_DATASET_PATH = 'attribute_dataset.csv'

TOP_MOVIE_COUNT = 100
MIN_RATINGS = 25

In [ ]:
def load_dataset(
    path: str = RATINGS_PATH,
    cols: Optional[list] = None,
    ):
  '''
  Loads dataset as DataFrame, filters to specified list of columns
  '''
  print(f"Loading {path} dataset...")
  dataset = pd.read_csv(path, low_memory=False)
  if cols:
    return dataset[cols]
  else:
    return dataset

In [ ]:
def print_frequencies(ratings: pd.DataFrame):
  '''
  Prints user and movie count
  '''
  users = ratings['userId'].unique().size
  movies = ratings['movieId'].unique().size
  print(f"Dataset contains {users} unique users and {movies} unique movies")

In [ ]:
def subset_ratings(
    ratings: pd.DataFrame,
    top_movie_count: int = TOP_MOVIE_COUNT,
    min_ratings: int = MIN_RATINGS
    ) -> pd.DataFrame:
  '''
  Subsets ratings dataset
  '''
  print(f"Subsetting ratings dataset...")
  movie_frequencies = ratings.groupby('movieId').count().sort_values('rating', ascending=False)
  ratings = ratings[ratings['movieId'].isin(movie_frequencies.index[:top_movie_count])]

  user_frequencies = ratings[ratings['rating']>=4].groupby('userId').count()['rating']
  ratings = ratings[ratings['userId'].isin(user_frequencies[user_frequencies>=min_ratings].index)]
  print_frequencies(ratings)
  return ratings

In [ ]:
def process_ratings(use_cache: bool = True) -> pd.DataFrame:
  '''
  Process the ratings dataset
  '''
  if use_cache and os.path.exists(RATINGS_SUBSET_PATH):
    ratings = load_dataset(RATINGS_SUBSET_PATH, cols=['userId', 'movieId', 'rating'])
  else:
    ratings = load_dataset(path=RATINGS_PATH, cols=['userId', 'movieId', 'rating'])
    print_frequencies(ratings)
    ratings = subset_ratings(ratings)
    ratings.to_csv(RATINGS_SUBSET_PATH)
  return ratings

In [ ]:
def get_popular_keywords(
    keywords: pd.DataFrame,
    k: int = 500
    ):
  '''
  Gets keywords that are found in at least k movie descriptions
  '''
  keyword_count = {}
  for keyword_list in tqdm(keywords['keywords']):
    for keyword in ast.literal_eval(keyword_list):
      if keyword['name'] not in keyword_count:
        keyword_count[keyword['name']] = [1]
      else:
        keyword_count[keyword['name']][0] += 1

  keyword_count = pd.DataFrame.from_dict(keyword_count, orient='index', columns=['k'])
  return np.array(keyword_count[keyword_count['k']>=20].index)

In [ ]:
def get_popular_names(
    credits: pd.DataFrame,
    target_col: str,
    n: int,
    ):
  '''
  Gets names of n most influential actors / directors
  '''
  name_weights = {}
  for row in tqdm(range(credits.shape[0])):
    for name in ast.literal_eval(credits.loc[row, target_col]):
      if target_col == 'cast' or (target_col == 'crew' and name['job'] == 'Director'):
        denominator = (name['order'] + 1) if target_col == 'cast' else 1
        weight = credits.loc[row, 'vote_count'] / denominator
        if name['name'] not in name_weights:
          name_weights[name['name']] = weight
        else:
          name_weights[name['name']] += weight

  name_weights = pd.DataFrame.from_dict(name_weights, orient='index', columns=['vote_count'])
  return name_weights.sort_values('vote_count', ascending=False).head(n).index

In [ ]:
def ohe_popular_attributes(
    attribute_dataset: pd.DataFrame,
    target_col: str,
    popular_attributes: list
    ):
  '''
  Generates one-hot encoded features for popular attributes
  '''

  features = np.zeros([attribute_dataset.shape[0], len(popular_attributes)], dtype=int)

  for row_index, json in enumerate(tqdm(attribute_dataset[target_col])):
    for attribute in ast.literal_eval(json):
      if attribute['name'] in popular_attributes:
        attribute_index = np.where(popular_attributes==attribute['name'])[0][0]
        features[row_index, attribute_index] = 1

  features = pd.DataFrame(features, columns=popular_attributes)
  return pd.concat([attribute_dataset['id'], features], axis=1)

In [ ]:
def parse_genres(metadata: pd.DataFrame) -> pd.DataFrame:
  '''
  Reformat genre data in columnar format
  '''
  genres = []
  for line in metadata['genres']:
    for genre in ast.literal_eval(line):
      if genre['name'] not in genres:
        genres.append(genre['name'])

  genres = np.array(genres[:20]) # remove the erroneous genres
  genre_matrix = np.zeros([metadata.shape[0], len(genres)], dtype=int)

  for row_index, line in enumerate(tqdm(metadata['genres'])):
    for genre in ast.literal_eval(line):
      if genre['name'] in genres:
        genre_index = np.where(genres==genre['name'])[0][0]
        genre_matrix[row_index, genre_index] = 1

  genre_matrix = pd.DataFrame(genre_matrix, columns=genres)
  metadata = pd.concat([metadata, genre_matrix], axis=1)
  metadata.drop('genres', axis=1, inplace=True)
  return metadata

In [ ]:
def merge_in_metadata(ratings: pd.DataFrame, use_cache: bool = True) -> pd.DataFrame:
  '''
  Reformat genre data in columnar format
  '''
  if use_cache and os.path.exists(FINAL_DATASET_PATH):
    ratings = load_dataset(FINAL_DATASET_PATH)
    ratings.pop(ratings.columns[0])
  else:
    metadata = load_dataset(path=METADATA_PATH, cols=['id', 'title', 'genres', 'revenue', 'vote_count']) # expand to other categories later
    metadata['id'] = pd.to_numeric(metadata['id'], errors='coerce')
    metadata = parse_genres(metadata)

    keywords = load_dataset(path=KEYWORDS_PATH)
    popular_keywords = get_popular_keywords(keywords)
    keyword_features = ohe_popular_attributes(attribute_dataset=keywords, target_col='keywords', popular_attributes=popular_keywords)

    credits = load_dataset(path=CREDITS_PATH)
    credits = credits.merge(metadata[['id', 'title', 'revenue', 'vote_count']], how='inner', on='id')
    popular_actors = get_popular_names(credits=credits, target_col='cast', n=50)
    # popular_actors = get_popular_names(credits=credits, target_col='cast', n=500)
    actor_features = ohe_popular_attributes(attribute_dataset=credits, target_col='cast', popular_attributes=popular_actors)
    popular_directors = get_popular_names(credits=credits, target_col='crew', n=10)
    director_features = ohe_popular_attributes(attribute_dataset=credits, target_col='crew', popular_attributes=popular_directors)

    metadata = metadata.merge(actor_features, how='inner', on='id')

    links = load_dataset(path=LINKS_PATH, cols=['movieId', 'tmdbId'])
    links.rename(columns={'tmdbId':'id'}, inplace=True)
    metadata = metadata.merge(links, how='inner', on='id').drop(columns=['revenue', 'vote_count'])
    ratings = ratings.merge(metadata.drop(columns=['id']), how='left', on='movieId').fillna(0)
    ratings.to_csv(FINAL_DATASET_PATH)

  return ratings

In [ ]:
def get_attribute_ratings(ratings: pd.DataFrame, k: int = 3, use_cache: bool = True) -> pd.DataFrame:
  '''
  Reformat genre data in columnar format
  '''
  if use_cache and os.path.exists(ATTRIBUTE_DATASET_PATH):
    attribute_ratings = load_dataset(ATTRIBUTE_DATASET_PATH)
    attribute_ratings.pop(attribute_ratings.columns[0])
    attribute_ratings['title'] = attribute_ratings['title'].astype('category')
  else:
    attribute_ratings = ratings.copy()
    _sum = attribute_ratings.groupby('userId').sum().copy()
    attribute_ratings.iloc[:, 4:] = attribute_ratings.iloc[:, 4:].multiply(
        attribute_ratings.iloc[:, 2],
        axis="index"
        )
    attribute_ratings.replace(0, np.nan, inplace=True)
    _median = attribute_ratings.groupby('userId').median().copy()
    attribute_ratings = ratings[['userId', 'title', 'rating']].copy()

    for c in tqdm(ratings.columns[4:]):
      new_data = _median[_sum[c] >= k].copy()
      new_data['rating'] = new_data[c]
      new_data = new_data[['movieId', 'rating']]
      new_data['title'] = c
      new_data = new_data.reset_index()
      new_data = new_data[['userId', 'title', 'rating']]
      attribute_ratings = pd.concat([attribute_ratings, new_data], axis=0)
      attribute_ratings['title'] = attribute_ratings['title'].astype('category')

    attribute_ratings.to_csv(ATTRIBUTE_DATASET_PATH)
  return attribute_ratings.reset_index(drop=True)

In [ ]:
# def pivot_data(attribute_ratings: pd.DataFrame):
#   '''
#   Binarises ratings and pivots the data
#   '''
#   attribute_ratings['rating'] = (attribute_ratings['rating'] >= 4).astype(int)
#   pivoted_ratings = pd.pivot_table(attribute_ratings, values='rating', index='userId', columns='title')

#   # return attribute_ratings['title'].cat.categories, pivoted_ratings
#   return pivoted_ratings

In [ ]:
# def prepare_data(pivoted_ratings):
#   '''
#   Generates RNN model inputs from the data
#   '''
#   # save original movie indexes and reindex
#   index_dict = copy.deepcopy(pivoted_ratings.columns)
#   pivoted_ratings.columns = list(range(len(index_dict)))

#   # create ordered index matrix
#   index_input_ordered = np.tile(pivoted_ratings.columns, (pivoted_ratings.shape[0], 1))
#   # na_filter = pivoted_ratings.notna()
#   # na_filter[na_filter==False] = None
#   # index_input_ordered = (index_input_ordered * na_filter).astype(float)

#   # shuffle the movies (random order, different for every user)
#   # index = (~np.isnan(pivoted_ratings).values * abs(np.random.randn(*pivoted_ratings.shape))).argsort(axis=1)
#   # index_input = np.nan_to_num(np.take_along_axis(index_input_ordered.values, index, axis=1), nan=len(index_dict))
#   # rating_input = np.nan_to_num(np.take_along_axis(pivoted_ratings.values, index, axis=1), nan=99)

#   index = (abs(np.random.randn(*pivoted_ratings.shape))).argsort(axis=1)
#   index_input = np.take_along_axis(index_input_ordered, index, axis=1)
#   rating_input = np.nan_to_num(np.take_along_axis(pivoted_ratings.values, index, axis=1), nan=2)
#   rating_input = np.eye(3)[rating_input.astype(int)] # one-hot encode positive/negative/not seen

#   output = np.tile(pivoted_ratings.values, (1, pivoted_ratings.shape[1])).reshape(*pivoted_ratings.shape[0:2], -1)
#   return index_dict, index_input, rating_input, output

In [ ]:
# def train_val_split(datasets, val_size=0.2):
#   '''
#   Split datasets into train and test (based on 0th dimension, which is user_id)
#   '''
#   np.random.seed(13)
#   rnd = np.random.rand(datasets[0].shape[0])
#   output = []
#   for d in datasets:
#     output.append(d[rnd > val_size])
#     output.append(d[rnd <= val_size])
#   return output

## Build a prediction (extrapolation) model
Create a recurrent neural network that ingests a movie-rating pair at every timestep and updates its internal representation of movie tastes of a given user. This model is used in the overall architecture to calculate reward (loss delta) for the policy network.

The recurrent network receives concatenated embeddings of movie indexes and corresponding ratings (shuffled across users) at each timestep and predicts ratings for all movies for a given user. It uses attention mechanism over inputs and outputs, as input order is scrambled. This allows the model to learn correlations between all observed movies.

In [ ]:
# def crossentropy_nan(y_true, y_pred):
#   '''
#   Implementation of crossentropy loss that masks NaN outputs
#   '''
#   y_pred = tf.where(tf.math.is_nan(y_true), tf.zeros_like(y_true), y_pred)
#   y_true = tf.where(tf.math.is_nan(y_true), tf.zeros_like(y_true), y_true)
#   return K.binary_crossentropy(y_true, y_pred)

In [ ]:
ratings = process_ratings()
ratings = merge_in_metadata(ratings)
movies = ratings['title'].unique()
attribute_ratings = get_attribute_ratings(ratings)
del ratings
# pivoted_ratings = pivot_data(attribute_ratings)
# del attribute_ratings

Loading ratings_subset.csv dataset...
Loading final_dataset.csv dataset...
Loading attribute_dataset.csv dataset...


In [ ]:
attribute_ratings.head()

,userId,title,rating
0,12,Toy Story,4.0
1,12,Twelve Monkeys,2.0
2,12,Babe,5.0
3,12,Se7en,5.0
4,12,The Usual Suspects,5.0


In [ ]:
# # def extrapolation_model(y_n=MOVIES_N, lr=0.01):
# def extrapolation_model(y_n=pivoted_ratings.shape[1], lr=0.001):
#   '''
#   Given movie vector dimension, generates the model object
#   '''
#   # encoder
#   index_input = Input((None,), name='IndexInput')

#   rating_input = Input((None, 3), name='RatingInput')
#   embedding = Embedding(y_n+1, y_n//2, name='MovieEmbedding') # shared throughout the network
#   X = embedding(index_input)
#   X = Concatenate(name="ConcatIndexAndRating")([X, rating_input])
#   encoder_out = LSTM(y_n//2, activation='sigmoid', return_sequences=True, name='RecurrentLayerEncoder')(X)

#   # decoder
#   X = TimeDistributed(Dense(y_n, activation='sigmoid'), name='Dense1')(encoder_out)
#   X = TimeDistributed(Dense(y_n//2, activation='sigmoid'), name='Dense2')(X)
#   attention = Attention(name='Attention')([encoder_out, X])
#   X = Concatenate(name="ConcatOutputAndAttention")([X, attention])
#   X = TimeDistributed(Dense(y_n, activation='sigmoid'), name='RatingPredictions')(X)

#   # build and compile model
#   model = Model(inputs=[index_input, rating_input], outputs=X, name='ExtrapolationModel')
#   model.compile(optimizer=Adam(lr=lr), loss=crossentropy_nan,  metrics=["accuracy"])
#   return model

In [ ]:
# extrapolation = extrapolation_model()
# extrapolation.summary()

In [ ]:
# index_dict, index_input, rating_input, output = prepare_data(pivoted_ratings)
# index_input_train, index_input_val, rating_input_train, rating_input_val, output_train, output_val = \
#   train_val_split([index_input, rating_input, output])
# del index_input
# del rating_input
# del output
# del pivoted_ratings
# checkpointer = ModelCheckpoint(filepath='../SimpleBots/2022/extrapolation_lstm_attr-230108-{epoch:02d}.hdf5',
#                                save_weights_only=True, verbose=1)

# extrapolation.load_weights('../SimpleBots/2022/extrapolation_lstm_attr-230108-100.hdf5')
# # extrapolation.fit([index_input_train, rating_input_train.reshape(*rating_input_train.shape, 1)], output_train, epochs=100, callbacks=[checkpointer], verbose=1)
# # extrapolation.load_weights('../../SimpleBots/2022/extrapolation_lstm_real_data_0316-300.hdf5')
# # extrapolation.load_weights("../../SimpleBots/2022/extrapolation_lstm_real_data_0316-300.hdf5")

# Pytorch model

In [ ]:
N_ITEMS = len(attribute_ratings['title'].unique())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class ExtrapolationModel(nn.Module):
    '''
    Builds a model given movie vector dimension
    '''
    # def __init__(self, y_n=pivoted_ratings.shape[1], lr=0.001):
    def __init__(self, y_n=N_ITEMS, lr=0.001):
        super(ExtrapolationModel, self).__init__()
        self.embedding = nn.Embedding(y_n+1, y_n//2) # embedding size seems high
        self.lstm = nn.LSTM(y_n//2+3, y_n//2, 1, batch_first=True, bidirectional=False)
        self.dense1 = nn.Linear(y_n//2, y_n) # what are params
        self.dense2 = nn.Linear(y_n, y_n//2)
        self.attention = nn.MultiheadAttention(y_n//2, num_heads=1)
        self.output = nn.Linear(y_n, y_n*2) # explicit and implicit model outputs stacked together
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, index_input, rating_input):

        # encoder
        print(f"{index_input.shape=}, {rating_input.shape=}")
        x = self.embedding(index_input)
        print(f"embedded {x.shape=}")
        x = torch.cat((x, rating_input), dim=-1)
        print(f"cat {x.shape=}")
        encoder_output, _ = self.lstm(x)
        print(f"lstm {encoder_output.shape=}")

        # decoder
        x = nn.ReLU()(self.dense1(encoder_output)) # check time distributed
        print(f"dense1 {x.shape=}")
        x = nn.ReLU()(self.dense2(x))
        print(f"dense2 {x.shape=}")
        attention, _ = self.attention(encoder_output, x, x) # attention of each input to output embedding
        print(f"attn {attention.shape=}")
        x = torch.cat((x, attention), dim=-1) # is there a need to combine pure lstm and attention?
        print(f"cat {x.shape=}")
        x = self.output(x)
        print(f"output {x.shape=}")
        return x

In [ ]:
from sklearn.model_selection import train_test_split

def train_val_split(dataset, val_size=0.2):
  '''
  Split datasets into train and test (based on user_id)
  '''
  random_state = 13

  train_ids, val_ids = train_test_split(
      dataset['userId'].unique(),
      test_size=val_size,
      random_state=random_state
      )

  train_dataset = dataset[dataset['userId'].isin(train_ids)]
  val_dataset = dataset[dataset['userId'].isin(val_ids)]

  return train_dataset, val_dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class CustomDataset(Dataset):
    def __init__(self, inputs):
        self.inputs = inputs

    def __len__(self):
        return len(self.inputs['userId'].unique())

    def __getitem__(self, index):

        user_data = self.inputs[self.inputs['userId']==self.inputs['userId'].unique()[index]]
        user_data = self.pivot_data(user_data)
        index_input, rating_input, output = self.prepare_data(user_data)
        return torch.LongTensor(index_input), torch.LongTensor(rating_input), torch.Tensor(output)

    def pivot_data(self, user_data: pd.DataFrame):
        '''
        Pivots the dataset
        '''
        user_data = pd.pivot_table(user_data, values='rating', index='userId', columns='title')
        user_data = user_data.reindex(columns=list(range(len(title_dict))))
        return user_data

    def prepare_data(self, user_data: pd.DataFrame):
        '''
        Generates model inputs and outputs from the data
        '''
        index_input = abs(np.random.randn(*user_data.columns.shape)).argsort()

        rating_input = np.nan_to_num(np.take_along_axis(user_data.values[0], index_input, axis=0), nan=2)
        rating_input = np.eye(3)[rating_input.astype(int)] # one-hot encode negative/positive/not seen

        explicit_output = np.tile(user_data.values, user_data.shape[1]).reshape(user_data.shape[1], -1)
        implicit_output = np.ones(explicit_output.shape)
        implicit_output[np.isnan(explicit_output)] = 0
        output = np.concatenate([explicit_output, implicit_output], axis=1)

        return index_input, rating_input, output

In [ ]:
import torch
import torch.nn as nn

class BCEWithLogitsLossNan(nn.Module):
    def forward(self, y_pred, y_true):
        """
        Implementation of BCEWithLogitsLoss that masks NaN outputs
        """
        mask = torch.isnan(y_true)
        y_pred = torch.where(mask, torch.zeros_like(y_true), y_pred)
        y_true = torch.where(mask, torch.zeros_like(y_true), y_true)
        return torch.nn.functional.binary_cross_entropy_with_logits(y_pred, y_true)

In [ ]:
class AccuracyNan(nn.Module):
    def forward(self, y_pred, y_true):
        """
        Implementation of accuracy that masks NaN outputs
        """
        mask = torch.isnan(y_true)
        y_pred = torch.where(mask, torch.zeros_like(y_true), y_pred)
        y_true = torch.where(mask, torch.zeros_like(y_true), y_true)
        y_pred_classes = y_pred >= 0.5
        return (y_pred_classes == y_true).float().mean().item()

In [ ]:
import copy
import logging
from typing import Optional, Tuple, Union
import pickle
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


class BinaryClassifierModel():
    def __init__(
        self,
        model: nn.Module,
        batch_size: int,
        learning_rate: float = 0.0001,
        epochs: int = 1000,
        optimiser: type = optim.Adam,
        # loss: nn.modules.loss._Loss = torch.nn.BCEWithLogitsLoss(),
        loss: nn.Module = BCEWithLogitsLossNan(),
        accuracy: nn.Module = AccuracyNan(),
        n_items: int = N_ITEMS,
        silent = False
    ):
        """
        Initialises the PyTorch based binary classifier object.

        :param model: PyTorch model object
        :param batch_size: batch size for model training
        :param learning_rate: training learning rate
        :param epochs: max number of training epochs
        :param optimiser: optimiser object
        :param loss: loss object
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.loss = loss.to(self.device)
        self.accuracy = accuracy.to(self.device)
        self.optimiser = optimiser(self.model.parameters(), lr=learning_rate)
        self.batch_size = batch_size
        self.epochs = epochs
        self.loss_history = {}
        self.n_items = n_items
        self.silent = silent

    def fit(
        self,
        train_data: pd.DataFrame,
        val_data: Optional[pd.DataFrame] = None,
        early_stopping_rounds: int = 10,
        save_weights: bool = False,
        epochs: Optional[int] = None,
    ) -> "BinaryClassifierModel":
        """
        Fits the binary classifier.

        :param train_data: training dataset
        :param val_data: validation dataset
        :param early_stopping_rounds: for how many epochs to train model after loss stops improving
        :param save_weights: flag for saving weights to disk
        :param epochs: max number of training epochs
        :return: self
        """

        train_iterator = self._dataset_iterator(train_data)
        best_loss_epoch = 0

        if val_data is not None:
            val_iterator = self._dataset_iterator(val_data)
            best_valid_loss = float("inf")
        else:
            best_train_loss = float("inf")

        if epochs:
            self.epochs = epochs

        for epoch in range(self.epochs):

            train_loss, _ = self._train(train_iterator)
            self.loss_history[epoch] = {}
            self.loss_history[epoch]['train'] = train_loss

            if val_data is not None:
                valid_loss, _ = self._evaluate(val_iterator)
                self.loss_history[epoch]['val'] = valid_loss
                if valid_loss < best_valid_loss:
                    best_valid_loss = valid_loss
                    best_loss_epoch = epoch
                    best_model = copy.copy(self.model)

            else:
                if train_loss < best_train_loss:
                    best_train_loss = train_loss
                    best_loss_epoch = epoch
                    best_model = copy.copy(self.model)


            if val_data is not None:
                print(f"\tVal. loss: {valid_loss:.3f}")

            if save_weights:
              now = datetime.datetime.now()
              weights_filename = now.strftime("recomender_weights_%Y_%m_%d_%H_%M_%S")
              history_filename = now.strftime("recomender_loss_history_%Y_%m_%d_%H_%M_%S.pickle")
              torch.save(self.model.state_dict(), weights_filename)
              with open(history_filename, 'wb') as handle:
                  pickle.dump(self.loss_history, handle, protocol=pickle.HIGHEST_PROTOCOL)

            if epoch >= best_loss_epoch + early_stopping_rounds:
                self.model = copy.copy(best_model)
                break

        return self

    def predict(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Returns predictions as a DataFrame.

        :param data: dataset
        :return: a DataFrame with predicted ratings
        """
        self.model.eval()

        predictions = np.empty([0, self.n_items*2])
        for (index_input, rating_input, _) in self._dataset_iterator(data):
            with torch.no_grad():
                index_input = index_input.to(self.device)
                rating_input = rating_input.to(self.device)
                outputs = self.model(index_input, rating_input).sigmoid()
                outputs = outputs[:, -1, :].detach().numpy()
                predictions = np.concatenate([predictions, outputs], axis=0)
        return predictions


    def param_count(self) -> int:
        """
        Returns number of trainable parameters in the model.

        :return: number of trainable parameters
        """
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)

    def _dataset_iterator(
        self,
        data: pd.DataFrame,
    ) -> DataLoader[object]:
        """
        Converts dataset into PyTorch DataLoader.

        :param data: dataset
        :return: DataLoader iterator object
        """
        dataset = CustomDataset(data)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False)

    def _train(self, iterator: DataLoader[object]) -> Tuple[float, float]:
        """
        Performs forward and backpropagation, updates model params
        based on the loss.

        :param iterator: dataset iterator
        :return: average epoch loss
        """
        epoch_loss = 0.0
        epoch_acc = 0.0
        self.model.train()

        t = tqdm.tqdm(iterator, position=0, leave=True, disable=self.silent)

        for (i, (index_input, rating_input, output)) in enumerate(t):
            index_input = index_input.to(self.device)
            rating_input = rating_input.to(self.device)
            output = output.to(self.device)
            self.optimiser.zero_grad()
            y_pred = self.model(index_input, rating_input)
            loss = self.loss(y_pred, output)
            loss.backward()
            self.optimiser.step()
            epoch_loss += loss.item()
            acc = self.accuracy(y_pred, output)
            epoch_acc += acc
            t.set_description(f"Train loss = {epoch_loss / (i+1):.3f}, Train accuracy = {epoch_acc / (i+1):.3f}")
        return epoch_loss / len(iterator), epoch_acc / len(iterator)

    def _evaluate(self, iterator: DataLoader[object]) -> Tuple[float, float]:
        """
        Performs forward propagation only and calculates loss
        without updating model params.

        :param iterator: dataset iterator
        :return: average epoch loss
        """
        epoch_loss = 0.0
        epoch_acc = 0.0
        self.model.eval()

        with torch.no_grad():
            for (index_input, rating_input, output) in iterator:
                index_input = index_input.to(self.device)
                rating_input = rating_input.to(self.device)
                output = output.to(self.device)
                y_pred = self.model(index_input, rating_input)
                loss = self.loss(y_pred, output)
                epoch_loss += loss.item()
                acc = self.accuracy(y_pred, output)
                epoch_acc += acc
        return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
attribute_ratings.columns = ['userId', 'title', 'rating']
attribute_ratings['title'] = attribute_ratings['userId'].astype('category')

In [ ]:
# move it to where this dataset is being prepared
attribute_ratings['userId'] = attribute_ratings['userId'].astype('category')

title_dict = attribute_ratings['title'].cat.categories
user_dict = attribute_ratings['userId'].cat.categories

# recode into numeric
attribute_ratings['title'] = attribute_ratings['title'].cat.codes
attribute_ratings['userId'] = attribute_ratings['userId'].cat.codes
attribute_ratings['rating'] = (attribute_ratings['rating'] >= 4).astype(int)

train_dataset, val_dataset = train_val_split(attribute_ratings)

In [ ]:
EPOCHS = 100
BATCH_SIZE = 1

model = ExtrapolationModel()

pipeline = BinaryClassifierModel(
          model=model,
          epochs=EPOCHS,
          batch_size=BATCH_SIZE,
)

In [ ]:
pipeline.fit(
      train_dataset,
      val_data=val_dataset,
      early_stopping_rounds=5,
      save_weights=True
)

  0%|          | 0/29711 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load('recomender_weights_2023_02_10_17_06_05'))

In [ ]:
val_dataset.tail()

In [ ]:
val_dataset[val_dataset['userId']==8].shape


In [ ]:
preds = pipeline.predict(val_dataset[val_dataset['userId']==8])[0]
# include tqdm for larger datasets

In [ ]:
def get_ground_truth_values(dataset):
  dataset_proc = CustomDataset(dataset)
  dataset = DataLoader(dataset_proc, batch_size=dataset.shape[0])
  for d in dataset:
    _, _, ground_truth = d
  # ground_truth = ground_truth[:, -1, :].reshape([-1, 130, 2])
  ground_truth = ground_truth[:, -1, :]
  return ground_truth

In [ ]:
gt = get_ground_truth_values(val_dataset[val_dataset['userId']==8])[0]

In [ ]:
gt

In [ ]:
preds >= 0.5

In [ ]:
np.abs(gt - (preds >= 0.5)).nansum() / 260 # error rate ??? hyperparams? run on more examples

### Model evaluation

In [ ]:
# helper functions for getting predictions and calculating loss after an arbitrary number of timesteps
def get_predictions(indexes, ratings, model=extrapolation):
  '''
  Based on ratings observed by QBot and extrapolation mode
  calculates predictions for the full rating vector
  '''
  # reshape inputs
  n_steps = len(indexes)
  indexes = np.array(indexes).reshape(1, -1)
  ratings = np.array(ratings).reshape(1, -1, 3)

  # evaluate cross-entropy loss
  predictions = extrapolation.predict([indexes, ratings], verbose=0)[0][-1]
  return predictions

def calculate_loss(predictions, ground_truth):
  '''
  Based on QBot's predictions and full list of ratings of ABot
  calculates extrapolation model loss (used for calculating reward)
  '''
  filt = ~np.isnan(np.array(ground_truth).reshape(-1, 1))
  loss = log_loss(np.array(ground_truth).reshape(-1, 1)[filt],
                  np.array(predictions, dtype=np.float64).reshape(-1, 1)[filt], labels=[0, 1])
  return loss

In [ ]:
# loss change on all outputs
import plotly.express as px

losses = np.zeros(130)
for i in tqdm(range(130)):
  predictions = extrapolation.predict([index_input_val[:1000, :i+1], rating_input_val[:1000, :i+1]], verbose=0)[:,-1,:]
  loss = calculate_loss(predictions, output_val[:1000, 0, :])
  losses[i] += loss

fig = px.line(y=losses, labels={"x": "Timestep", "y": "Loss"})
fig.show()

In [ ]:
# loss change on holdout
index_input_partial = index_input_val.copy()
rating_input_partial = rating_input_val.copy()

index_input_partial[index_input_partial > 65] = 130
rating_input_partial[index_input_partial > 65] = [0, 0, 1]

resorted_idx = (index_input_partial==130).argsort()
index_input_partial = np.take_along_axis(index_input_partial, resorted_idx[:, ::-1], axis=1)
rating_input_partial = np.take_along_axis(rating_input_partial, np.expand_dims(resorted_idx[:, ::-1], axis=2), axis=1)

losses = np.zeros(66)
for i in tqdm(range(64, 130, 1)):
  predictions = extrapolation.predict([index_input_val[:1000, 64:i+1], rating_input_val[:1000, 64:i+1]], verbose=0)[:,-1,:]
  loss = calculate_loss(predictions[:, 65:], output_val[:1000, 0, 65:])
  losses[i-64] += loss

fig = px.line(y=losses, labels={"x": "Timestep", "y": "Loss"})
fig.update_yaxes(rangemode="tozero")
fig.show()

In [ ]:
# check if attributes can predict movie ratings
attribute_indexes = [i for i, v in enumerate(index_dict) if v not in movies]
movie_indexes = [i for i, v in enumerate(index_dict) if v in movies]

index_input_partial = index_input_val.copy()
rating_input_partial = rating_input_val.copy()

index_input_partial[np.isin(index_input_partial, movie_indexes)] = 130
rating_input_partial[np.isin(index_input_partial, movie_indexes)] = [0, 0, 1]

resorted_idx = (index_input_partial==130).argsort()
index_input_partial = np.take_along_axis(index_input_partial, resorted_idx[:, ::-1], axis=1)
rating_input_partial = np.take_along_axis(rating_input_partial, np.expand_dims(resorted_idx[:, ::-1], axis=2), axis=1)

losses = np.zeros(30)

for i in tqdm(range(100, 130, 1)):
    predictions = extrapolation.predict([index_input_partial[:1000, 100:i+1], rating_input_partial[:1000, 100:i+1]], verbose=0)[:,-1,:]
    loss = calculate_loss(predictions[:, movie_indexes], output_val[:1000, 0, movie_indexes])
    losses[i-100] += loss

fig = px.line(y=losses, labels={"x": "Timestep", "y": "Loss"})
fig.update_yaxes(rangemode="tozero")
fig.show()

100%|██████████| 30/30 [00:12<00:00,  2.35it/s]


In [ ]:
# feature importance - loss change after single input

index_input_partial = index_input_val.copy()
rating_input_partial = rating_input_val.copy()


resorted_idx = index_input_partial.argsort()
index_input_partial = np.take_along_axis(index_input_partial, resorted_idx[:, :], axis=1)
rating_input_partial = np.take_along_axis(rating_input_partial, np.expand_dims(resorted_idx[:, :], axis=2), axis=1)

losses = np.zeros(130)
for i in tqdm(range(130)):
  predictions = extrapolation.predict([index_input_partial[:1000, i:i+1], rating_input_partial[:1000, i:i+1]], verbose=0)[:,-1,:]
  loss = calculate_loss(predictions[:, movie_indexes], output_val[:1000, 0, movie_indexes])
  losses[i] += loss

100%|██████████| 130/130 [00:22<00:00,  5.80it/s]


In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
feature_importance = pd.DataFrame(losses, index=index_dict, columns=['Loss'])
feature_importance.sort_values(by='Loss')

,Loss
title,
Groundhog Day,2.100008
Saving Private Ryan,2.117905
Russell Crowe,2.139066
Robert De Niro,2.147282
Fargo,2.147451
GoodFellas,2.161505
Braveheart,2.170456
L.A. Confidential,2.193690
The Shawshank Redemption,2.194281


In [ ]:
movie_importance = np.mean([feature_importance.loc[m, :].values[0] for m in movies])
attribute_importance = np.mean([feature_importance.loc[m, :].values[0] for m in feature_importance.index if m not in movies])
print(movie_importance, attribute_importance)

2.464109016771595 2.439663128728819


In [ ]:
import plotly.express as px
import pandas as pd

In [ ]:
loss = [2.464109016771595, 2.439663128728819]

In [ ]:
_labels={"x": "Timestep", "y": "Loss"}

In [ ]:
fig = px.bar(y=loss, labels={"x": "Entity Type, 0=movie, 1=attribute", "y": "Loss", 0:'Movies'})
fig.update_yaxes(rangemode="tozero")
fig.show()

In [ ]:
feature_importance['Loss'].describe()

count    130.000000
mean       2.458468
std        0.276290
min        2.100008
25%        2.274907
50%        2.405248
75%        2.544075
max        3.934081
Name: Loss, dtype: float64

In [ ]:
rating_input_partial.shape

(7403, 130, 3)

In [ ]:
np.expand_dims(np.array([[1, 0, 0]]), 1).shape

(1, 1, 3)

In [ ]:
entity = 'Russell Crowe'
index = np.array([[list(index_dict).index(entity)]])
liked = np.expand_dims(np.array([[1, 0, 0]]), 1)
disliked = np.expand_dims(np.array([[0, 1, 0]]), 1)
not_watched = np.expand_dims(np.array([[0, 0, 1]]), 1)

liked_preds = np.squeeze(extrapolation.predict([index, liked], verbose=0))
disliked_preds = np.squeeze(extrapolation.predict([index, disliked], verbose=0))
nw_preds = np.squeeze(extrapolation.predict([index, not_watched], verbose=0))

In [ ]:
diffs = pd.DataFrame(liked_preds - disliked_preds, index=index_dict, columns=['Diff'])
diffs.sort_values(by='Diff')

,Diff
title,
The Silence of the Lambs,-9.998884e-01
Crime,-9.946811e-01
Russell Crowe,-9.894220e-01
Apollo 13,-9.429434e-01
Adventure,-9.426590e-01
The Princess Bride,-9.358981e-01
The Godfather,-9.246876e-01
The Mask,-9.189034e-01
A Beautiful Mind,-9.101998e-01


In [ ]:
diffs = pd.DataFrame(liked_preds - nw_preds, index=index_dict, columns=['Diff'])
diffs.sort_values(by='Diff')

,Diff
title,
The Silence of the Lambs,-9.998692e-01
Russell Crowe,-9.904172e-01
Crime,-9.894108e-01
Jurassic Park,-9.648043e-01
The Princess Bride,-9.475608e-01
Apollo 13,-9.457770e-01
Adventure,-9.426590e-01
A Beautiful Mind,-9.119993e-01
Gladiator,-9.091681e-01


In [ ]:
# calculate average loss at each timestep
losses = []
for t in range(1, 21):
  predictions = extrapolation.predict([index_input_val[:, -t:], np.expand_dims(rating_input_val[:, -t:], axis=2)])[:, -1, :]
  losses_t = []
  for u in range(output_val.shape[0]):
    losses_t.append(calculate_loss(predictions[u], output_val[u, -1, :]))
  losses.append(np.mean(losses_t))

fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=np.array(range(20)), y=losses, name='loss'), secondary_y=False)
fig.add_trace(go.Scatter(x=np.array(range(20)), y=(np.sum(index_input_val[:, :]<20, axis=0) / index_input_val.shape[0])[::-1], name='users with ratings'), secondary_y=True )
fig.update_layout(title='Loss Decrease by Timestep', xaxis_title="Timestep", yaxis_title="Loss", width=800, height=500)
fig.show()

In [ ]:
# calculate average loss at each timestep
losses = []
for t in range(1, 21):
  predictions = extrapolation.predict([index_input_partial[:, -t:], np.expand_dims(rating_input_partial[:, -t:], axis=2)])[:, -1, :]
  losses_t = []
  for u in range(pivoted_ratings.shape[0]):
    if ~np.isnan(output[u, -1, 10:]).all():
      losses_t.append(calculate_loss(predictions[u, 10:], output[u, -1, 10:]))
  losses.append(np.mean(losses_t))

fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=np.array(range(20)), y=losses, name='loss'), secondary_y=False)
fig.add_trace(go.Scatter(x=np.array(range(20)), y=(np.sum(index_input_val[:, :]<20, axis=0) / index_input_val.shape[0])[::-1], name='users with ratings'), secondary_y=True )
fig.update_layout(title='Loss Decrease by Timestep', xaxis_title="Timestep", yaxis_title="Loss", width=800, height=500)
fig.show()

In [ ]:
1

In [ ]:
# sense check the model
movie_to_predict = 17
strongly_correlated = [5, 16]
weakly_correlated = [0, 13]

def demo_preds(other_movies, ratings=1, movie_to_predict=movie_to_predict):
  likes = '' if ratings==1 else 'dis'

  pred = extrapolation.predict([np.reshape(other_movies, (1, -1)),
                              np.reshape(np.tile(ratings, len(other_movies)), (1, -1, 1))])[0][-1][movie_to_predict]
  print(f"If user {likes}likes these movies, the model predicts the chance of the user liking '{index_dict[movie_to_predict]}' is {pred:.2f}")

print("The 'Star Wars' movies form a clear cluster in the data, so let's check if the model picked up on that.")
print(f"Let's use the model to predict the rating of movie '{index_dict[movie_to_predict]}'\n")

for m in strongly_correlated:
  print(f"Correlation with '{index_dict[m]}' is {(corr.iloc[movie_to_predict, m]):.2f}.")
demo_preds(strongly_correlated, ratings=1)
demo_preds(strongly_correlated, ratings=0)
print('')

for m in weakly_correlated:
  print(f"Correlation with '{index_dict[m]}' is {(corr.iloc[movie_to_predict, m]):.2f}.")
demo_preds(weakly_correlated, ratings=1)
demo_preds(weakly_correlated, ratings=0)

In [ ]:
# evaluate model's ability to predict ratings of a single movie based on an observed rating of another movie

def index_predict(target, predictor):
  '''
  Calculate cross-entropy loss of the model when rating of the predictor
  movie is used to predict the rating of the target movie
  '''
  seen_target = ~np.isnan(output_val[:, -1, target])
  seen_predictor = ~np.isnan(output_val[:, -1, predictor])
  filt = seen_target & seen_predictor
  indexes = np.expand_dims(np.tile(predictor, filt.sum()), axis=1)
  values = np.expand_dims(output_val[:, -1, predictor][filt], axis=[1, 2])
  predictions = np.array(np.expand_dims(extrapolation.predict([indexes, values])[:, -1, target], axis=1), dtype=np.float64)
  output = np.expand_dims(output_val[:, -1, target][filt], axis=1)
  loss = log_loss(output, predictions, labels=[0, 1])
  return loss

episodeIV = 5
losses = []
for i in range(20):
  losses.append(index_predict(episodeIV, i))

fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=index_dict, y=losses, name='loss'), secondary_y=False)
fig.add_trace(go.Scatter(x=index_dict, y=corr.iloc[5].reindex(index_dict), name='correlation'), secondary_y=True )
fig.update_layout(title=f"Pearson's Correlations and Model Loss for '{index_dict[episodeIV]}'", yaxis_title="Loss", width=1000, height=600)
fig.show()

## Build a policy model
Create a recurrent neural network that ingests a movie-rating pair at every timestep and outputs a probability for a given action using REINFORCE algorithm. Actions are defined as QBot's questions about specific movies in the list.

<img src="https://miro.medium.com/max/1216/1*evUko5QpAcvv-83xw0DP-g.jpeg" width="200">

In [ ]:
def loss_wrapper(index, y_n=MOVIES_N):
  '''
  Given rewards, predictions and action index implements REINFORCE algorithm.
  '''
  def reinforce_loss(rewards, y_pred):
    eps=1e-5
    y_pred = K.maximum(K.minimum(y_pred, (1-eps)), eps)
    chosen_action = K.sum(K.one_hot(index, y_n) * y_pred)
    loss = -K.log(chosen_action) * rewards
    return loss
  return reinforce_loss

In [ ]:
# def policy_model(a_n=MOVIES_N, layers=[120, 60, 10], lr=0.01):
def policy_model(a_n=MOVIES_N, layers=[100, 50], lr=0.01):
# def policy_model(a_n=MOVIES_N, layers=[20, 10], lr=0.001):
  '''
  Given movie vector dimension, generates the model object
  '''
  X_input = Input((a_n*3,))
  index = Input((1,), dtype='int32')
  X = X_input
  for l in layers:
    X = Dense(l, activation='sigmoid')(X)
  X = Dense(a_n, activation='softmax')(X)
  model = Model(inputs=[X_input, index], outputs=X, name='PolicyModel')
  model.compile(optimizer=Adam(learning_rate=lr), loss=loss_wrapper(index))
  return model

In [ ]:
policy = policy_model()
policy.summary()
# policy.load_weights("../../SimpleBots/policy_real_data.hdf5")
policy.load_weights("../../SimpleBots/2022/policy_real_data_0314.hdf5")
# policy.load_weights("../../SimpleBots/2022/policy_real_data_0317.hdf5")

# Generate dialogue

In [ ]:
def dialogue_dataset(ground_truth, movies=movies):
  '''
  Given binary matrix and a list of movie names generates dialogue dataset
  '''
  dataset = []
  for j, u in enumerate(ground_truth):
    user = {}
    for i, m in enumerate(movies):
      if ~np.isnan(ground_truth[j, i]):
        user[m] = 'liked' if u[i] == 1 else 'disliked'
    dataset.append(user)
  return dataset

In [ ]:
# sense check
ground_truth = pivoted_ratings.values
print("Input:", ground_truth[13])
print("Ouput:", dialogue_dataset(ground_truth)[13])

## Extending World class
Environment where bots act and observe each other's actions

In [ ]:
class QAWorld(DialogPartnerWorld):
  def __init__(self, opt, qbot, abot, shared=None):
    self.id = 'QAWorld'
    self.qbot = qbot
    self.abot = abot
    self.acts = []
    super(QAWorld, self).__init__(opt, [self.qbot, self.abot], shared)
    self.qbot.initialise(self.abot.binary_ratings)


  def parley(self):
    '''
    Implemets key functionality of the dialogue environment
    '''
    # QBot asks a question and observes it as well
    self.qbot.question = self.qbot.act()
    self.qbot.observe({
      'text': self.qbot.question['text'],
      'id': self.qbot.id
    })

    # ABot observes question, acts and observes its own answer
    self.abot.observe({
      'text': self.qbot.question['text'],
      'id': self.qbot.id,
    })
    self.abot.answer = self.abot.act()
    self.qbot.observe(self.abot.answer)
    self.acts = [self.qbot.question, self.abot.answer]


  def printout(self):
    '''
    Print out inputs, outputs and internal representations of both models
    '''
    filt = movies.isin(self.abot.ratings).values

    if len(self.qbot.old_representation) == 0:
      print("\n[ABot]'s ground truth ratings:", end='\n\t {')
      for i, k in enumerate(self.abot.ratings):
        e = ", " if i < len(self.abot.ratings) - 1 else "}\n\n"
        print(f'{k}: {self.abot.ratings[k]}', end=e)

    print("Step 1: [QBot]'s policy network makes a prediction:", end='\n\t {')
    for i, m in enumerate(movies):
      e = ", " if i < len(movies) - 1 else "}\n"
      print(f'{m}: {np.int(np.round((self.qbot.action_probs[i]*100)))}%', end=e)

    strategy = "explore" if self.qbot.exploration else "exploit"
    print(f"Step 2: [QBot] chooses a movie to ask about: {self.qbot.question['text']},",
          f"{np.int(np.round((self.qbot.action_probs[self.qbot.choice]*100)))}% probability ({strategy})")

    print(f"Step 3. [ABot] responds: {self.abot.answer['text']}")

    print("Step 4. [QBot] observes the dialogue round. [QBot] knows about these movies:", end='\n\t {')
    for i, k in enumerate(self.qbot.representation):
      e = ", " if i < len(self.qbot.representation) - 1 else "}\n"
      print(f'{k}: {self.qbot.representation[k]}', end=e)

    print("Step 5. [QBot]'s extrapolation network predicts the full vector of [ABot]'s ratings:", end='\n\t {')
    for i, m in enumerate(movies[filt]):
      e = ", " if i < len(movies[filt]) - 1 else "}\n"
      print(f'{m}: {str(np.round(self.qbot.predictions.ravel()[i], 2))}', end=e)

    print("Step 6. Prediction error is computed.\n\t", end='')
    print("Difference between predictions and ground-truth values is:", end='\n\t {')
    binary_ratings = self.abot.binary_ratings


    # DEBUG HERE: difference seems off

    pairwise_diff = (self.abot.binary_ratings - self.qbot.predictions.ravel())[filt]
    for i, m in enumerate(movies[filt]):
      e = ", " if i < len(movies[filt]) - 1 else "}\n\t"
      print(f'{m}: {str(np.round(pairwise_diff[i], 2))}', end=e)
    print(f"Log loss is: {str(np.round(self.qbot.error, 3))}")

    print(f"Step 7. Reward is computed as log loss improvement compared to the previous dialogue round: {str(np.round(self.qbot.reward, 3))}.")

    print(f"Step 8. [QBot]'s policy network weights are updated using REINFORCE algorithm. Model uses:", end='\n\t ')
    print(f"(a) State: [QBot]'s observations at the beginning of the round: ", end='[') #enum names
    for i, m in enumerate(self.qbot.old_representation):
      e = ", " if i < len(self.qbot.old_representation) - 1 else ""
      print(m, end=e)
    print(end="]\n\t ")
    print(f"(b) Action: index of the movie that [QBot] asked about: {movies.iloc[self.qbot.choice]}", end='\n\t ')
    print(f"(c) Reward: {str(np.round(self.qbot.reward, 3))}", end='\n\t')
    print(f"Policy loss is: {str(np.round(self.qbot.policy_loss, 3))}", end='\n\n')

## Extending Agent classes
Implementing bot-specific functionality

In [ ]:
class QBot(Agent):
  def __init__(self, opt, train_policy=False, epsilon=0.2, random_policy=False):
    self.id = 'QBot'
    self.train_policy = train_policy
    self.random_policy = random_policy
    self.epsilon = epsilon
    self.key = None
    self.abot_binary_ratings = []
    self.representation = {}
    self.old_representation = {}
    self.binary_representation = np.zeros(MOVIES_N*3)
    self.question = None
    self.error = None
    self.cum_rewards = []
    self.reward = None
    self.choice = None
    self.action_probs = []
    self.exploration = None
    #self.predictions = np.tile(0.5, MOVIES_N)
    self.predictions = get_predictions([20], [99])
    self.policy_loss = None
    self.indexes = []
    self.ratings = []


  def initialise(self, observation):
    '''
    Observes ground truth ratings at the beginning of an episode
    '''
    self.abot_binary_ratings = observation
    self.error = calculate_loss(self.predictions, self.abot_binary_ratings)


  def observe(self, observation):
    '''
    Stores observations for later and updates the policy model if train_policy flag is set to True
    '''
    self.observation = observation
    # observe own question
    if self.observation['id'] == self.id:
      self.key = self.observation['text']
      if self.key not in self.representation:
        self.indexes.append(np.argwhere(movies.values==self.key).item())

    # observe ABot's response
    else:
      if self.key not in self.representation:
        if self.observation['text'] != "haven't seen":
          self.ratings.append(1 if self.observation['text'] == 'liked' else 0)
        else:
          self.indexes.pop()
      self.calculate_reward()
      self.old_representation = self.representation.copy()
      self.representation[self.key] = self.observation['text']
      old_binary_representation = self.binary_representation
      self.binarise_representation()

      # update the policy model
      if self.train_policy:
        policy.fit(x=[np.reshape(old_binary_representation, (1, -1)), np.reshape(self.choice, (1, -1))],
                   y=np.reshape(self.reward, (1, -1)), verbose=0)

      # calculate policy loss
      self.policy_loss = policy.evaluate(x=[np.reshape(old_binary_representation, (1, -1)), np.reshape(self.choice, (1, -1))],
                                         y=np.reshape(self.reward, (1, -1)), verbose=0)


  def act(self):
    '''
    Returns QBot's question (title of a movie)
    '''
    self.choose_action()
    question = movies.iloc[self.choice]
    return {'id': self.id, 'text': question}


  def binarise_representation(self):
    '''
    Represents tasks in a way accessible to the neural net
    '''
    self.binary_representation = []
    for m in movies:
      movie_representation = [0, 0, 0]
      if m in self.representation:
        if self.representation[m] == 'liked':
          movie_representation = [1, 0, 0]
        elif self.representation[m] == 'disliked':
          movie_representation = [0, 1, 0]
        elif self.representation[m] == "haven't seen":
          movie_representation = [0, 0, 1]
      self.binary_representation += movie_representation


  def calculate_reward(self):
    '''
    Calculates the reward based on delta od errors of two consecutive conversational turns
    '''
    old_error = self.error
    if len(self.indexes) > 0:
      self.predictions = get_predictions(self.indexes, self.ratings)
    self.error = calculate_loss(self.predictions, self.abot_binary_ratings)
    self.reward = max(0, old_error - self.error)


  def choose_action(self):
    '''
    Chooses action proportionally to policy output, with epsilon-greedy exploration
    '''
    self.action_probs = policy.predict([np.reshape(self.binary_representation, (1, -1)), np.reshape(0, (1, -1))]).ravel()

    if self.random_policy:
      self.exploration = True
      self.choice = np.random.choice(len(movies))
    else:
      if not self.train_policy:
        self.exploration = False
        # self.choice = np.argmax(self.action_probs)
        self.choice = np.random.choice(len(movies), p=self.action_probs)
      else:
        if (np.random.rand() > self.epsilon):
          self.exploration = False
          self.choice = np.random.choice(len(movies), p=self.action_probs)
        else:
          self.exploration = True
          self.choice = np.random.choice(len(movies))

In [ ]:
class ABot(Agent):
  def __init__(self, opt, ratings):
    self.id = 'ABot'
    self.ratings = ratings
    self.binary_ratings = []
    self.answer = None
    self.binarise_ratings()


  def observe(self, observation):
    '''
    Stores observations for later
    '''
    self.observation = observation


  def act(self):
    '''
    Returns ABot's response (rating for the requested movie)
    '''
    if self.observation['text'] in self.ratings:
      answer = self.ratings[self.observation['text']]
    else:
      answer = "haven't seen"
    return {'id': self.id, 'text': answer}


  def binarise_ratings(self):
    '''
    Represents tasks in a way accessible to the neural net
    '''
    for m in movies:
      if m in self.ratings:
        self.binary_ratings.append(1 if self.ratings[m] == 'liked' else 0)
      else:
        self.binary_ratings.append(math.nan)

In [ ]:
def train_policy(dataset):
  '''
  Train policy network through bot-play
  '''
  opt = ParlaiParser().parse_args([])
  for i, r in enumerate(dataset):
    qbot = QBot(opt, train_policy=True)
    abot = ABot(opt, r)
    world = QAWorld(opt, qbot, abot)
    num_turns = 5 + ((len(dataset) - i) // (len(dataset) // 5))
    for j in range(num_turns):
      world.parley()
    if (i+1) % 1000 == 0:
      print(f"Step {i+1} of {len(dataset)}. Saving weights.")
      policy.save_weights(f"../../SimpleBots/2022/policy_real_data_0317_{i+1}.hdf5")

dataset = dialogue_dataset(output_train[:, -1, :])
train_policy(dataset)

# Dialogue Demo

In [ ]:
opt = ParlaiParser().parse_args([])
dataset = dialogue_dataset(output_val[:, -1, :])
example = np.random.choice(10000)
# example = 75
num_turns = 10
qbot = QBot(opt)
abot = ABot(opt, dataset[example])
world = QAWorld(opt, qbot, abot)
print("\n")
for i in range(num_turns):
    world.parley()
    print(f"***DIALOGUE ROUND {i+1}***")
    world.printout()

### Model evaluation

In [ ]:
def get_loss(dataset, random_policy=False):
  '''
  Evaluate policy network through bot-play
  '''
  opt = ParlaiParser().parse_args([])
  losses = []
  for i, r in enumerate(dataset):
    qbot = QBot(opt, random_policy=random_policy)
    abot = ABot(opt, r)
    world = QAWorld(opt, qbot, abot)
    num_turns = 20
    losses_t = []
    for j in range(num_turns):
      world.parley()
      losses_t.append(qbot.error)
    losses.append(losses_t)
    if (i+1) % 1000 == 0:
      print(f"Step {i+1} of {len(dataset)}.")
  return losses

dataset = dialogue_dataset(output_val[:, -1, :])
losses_trained_policy = get_loss(dataset)
losses_random_policy = get_loss(dataset, random_policy=True)

In [ ]:
trained_policy_by_timestep = np.array(losses_trained_policy).mean(axis=0)
random_policy_by_timestep = np.array(losses_random_policy).mean(axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(20)), y=trained_policy_by_timestep, name='trained policy net', mode='lines'))
fig.add_trace(go.Scatter(x=np.array(range(20)), y=random_policy_by_timestep, name='random baseline', mode='lines'))
fig.update_layout(title='Loss Decrease by Conversational Turn', xaxis = dict(dtick=1), xaxis_title="Conversational Turns", yaxis_title="Loss", width=800, height=500)
fig.show()